In [1]:
# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models
import tensorflow as tf
import random

from destruction_utilities import *
from destruction_statistics import *
from numpy import random
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks, preprocessing
from tensorflow.keras.utils import Sequence
from tensorflow.keras.metrics import CategoricalAccuracy, Precision, AUC
from tensorflow.keras.models import load_model
from sklearn.metrics import precision_recall_curve, roc_auc_score
from os import path
import zarr

In [2]:
CITY = 'aleppo'

In [ ]:
def get_zarr(city, type, set):
    path = f'../data/{city}/others/{city}_{type}s_{set}.zarr'
    return zarr.open(path, mode='r')

In [ ]:
# train_path = f'../data/{CITY}/others/{CITY}_images_train.zarr'
# validate_path = f'../data/{CITY}/others/{CITY}_images_validate.zarr'
# test_path = f'../data/{CITY}/others/{CITY}_images_test.zarr'

In [ ]:
class ZarrGenerator(Sequence):
    def __init__(self, images, labels, batch_size=32):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images)//self.batch_size
    
    def __getitem__(self, index):
        
#         n = self.images.shape[0]
#         indices = np.random.choice(np.arange(n),32)
        
#         X = self.images[indices]
#         y = self.labels[indices]
        X = self.images[index*self.batch_size:(index+1)*self.batch_size]
        y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        return self.augment(X), y.flatten()
    
    def augment(self, X):
        # Horizontal and vertical flip
        flipping_funcs = [
            lambda image: image,
            lambda image: np.fliplr(image),
            lambda image: np.flipud(image),
            lambda image: np.flipud(np.fliplr(image))
        ]
        func = random.choice(flipping_funcs)
        X = func(X)
        
        # Brightness
        alpha = random.choice(np.linspace(0.85, 1.4))
        X = X * alpha
        
        return X

In [ ]:
training_images = get_zarr(CITY, 'image', 'train')
training_labels = get_zarr(CITY, 'label', 'train')
validation_images = get_zarr(CITY, 'image', 'validate')
validation_labels = get_zarr(CITY, 'label', 'validate')
test_images = get_zarr(CITY, 'image', 'test')
test_labels = get_zarr(CITY, 'label', 'test')

In [ ]:
print(training_images.shape, training_labels.shape, np.unique(training_labels, return_counts=True))
print(validation_images.shape, validation_labels.shape, np.unique(validation_labels, return_counts=True))
print(test_images.shape, test_labels.shape, np.unique(test_labels, return_counts=True))


In [ ]:
from keras import backend as K

def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

auc = AUC(
    num_thresholds=200,
    curve='ROC',
)

In [ ]:
training_generator = ZarrGenerator(training_images, training_labels, batch_size=32)
validation_generator = ZarrGenerator(validation_images, validation_labels, batch_size=32)
test_generator = ZarrGenerator(test_images, test_labels, batch_size=32)

In [ ]:
# https://datascience.stackexchange.com/questions/45165/how-to-get-accuracy-f1-precision-and-recall-for-a-keras-model
# Callbacks
training_callbacks = [
    callbacks.EarlyStopping(monitor='val_auc', restore_best_weights=True, patience=5),
    callbacks.BackupAndRestore(backup_dir='../models')
]

args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
model = models.convolutional_network(**args)
model.compile(optimizer='adam', loss='binary_focal_crossentropy', metrics=['accuracy', f1_m, precision_m, recall_m, auc ])


# Train model on dataset
model.fit_generator(
    training_generator,
    validation_data=validation_generator, 
    epochs=100, 
    callbacks = training_callbacks
)


model.save(f'../models/{CITY}_unbalanced_auc', save_format="h5")
# train_generator.__getitem__(1)[0].shape

In [ ]:
model = load_model(f'../models/{CITY}_unbalanced')

In [ ]:
import gc
gc.collect()

batch_size = 5000
iters = test_images.shape[0] // batch_size
preds = []
labels = []
for i in range(0, iters):
    end = (i+1)*batch_size
    print(i)
    print(i*batch_size, end, "\n")
    if i == iters - 1:
        preds.append(model.predict(test_images[i*batch_size:]))
        labels.append(test_labels[i*batch_size:])
    else:
        preds.append(model.predict(test_images[i*batch_size: end]))
        labels.append(test_labels[i*batch_size: end])

In [ ]:
yhat = np.squeeze(np.concatenate(preds, axis=0))
y = np.squeeze(np.concatenate(labels, axis=0 ))
roc_auc_score(y, yhat)

In [ ]:
a = get_zarr(CITY, 'image', 'train')

In [ ]:
a.shape

In [ ]:
import time

start = time.time()

training_generator.__getitem__(0)

In [ ]:
n = training_images.shape[0]
indices = np.random.choice(np.arange(n),32)
        
# X = self.images[indices]
# y = self.labels[indices]

indices

In [ ]:
training_images[indices]